In [16]:
def BronzeTablesToVol(
    src_catalog: str,
    src_schema: str,
    tgt_catalog: str,
    tgt_schema: str,
    tgt_volume: str,
    folder_name: str,
    tbl_list: str
):
    tables_df = spark.sql(f"SHOW TABLES IN {src_catalog}.{src_schema}") \
        .filter("lower(tableName) <> 'dbtools_execution_history'")

    if tbl_list.strip() != "":
        tables_df = tables_df.filter(f"lower(tableName) in ({tbl_list})")

    for r in tables_df.collect():
        table_name = r["tableName"]
        full_name = f"{src_catalog}.{src_schema}.{table_name}"
        df = spark.read.table(full_name)
        # Write delta table directly to the external volume
        EXTERNAL_VOLUME_PATH = f"/Volumes/{tgt_catalog}/{tgt_schema}/{tgt_volume}/{folder_name}/{table_name}"
        df=spark.sql(f"select * from {src_catalog}.{src_schema}.{table_name}")
        df.write.format("delta").mode("overwrite").save(EXTERNAL_VOLUME_PATH)
        print(f"Table copied to={EXTERNAL_VOLUME_PATH}")

In [19]:
output_param_key= "TGT_TYPE"
tgt_type = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "VOL")
print("Param {} value is {}".format(output_param_key, tgt_type))

output_param_key= "SRC_CATALOG"
src_catalog = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "psftcln")
print("Param {} value is {}".format(output_param_key, src_catalog))

output_param_key= "SRC_SCHEMA"
src_schema = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "psftcln")
print("Param {} value is {}".format(output_param_key, src_schema))

output_param_key= "TGT_CATALOG"
tgt_catalog = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "default")
print("Param {} value is {}".format(output_param_key, tgt_catalog))

output_param_key= "TGT_SCHEMA"
tgt_schema = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "default")
print("Param {} value is {}".format(output_param_key, tgt_schema))

output_param_key= "TGT_VOLUME"
tgt_volume = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "psftos")
print("Param {} value is {}".format(output_param_key, tgt_volume))

output_param_key= "FOLDER_NAME"
folder_name = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "bronze")
print("Param {} value is {}".format(output_param_key, folder_name))

output_param_key= "TBL_LIST"
tbl_list = oidlUtils.parameters.getTaskValue("SET_PARAM", output_param_key, "")
print("Param {} value is {}".format(output_param_key, tbl_list))

Param TGT_TYPE value is VOL
Param SRC_CATALOG value is psftcln
Param SRC_SCHEMA value is psftcln
Param TGT_CATALOG value is default
Param TGT_SCHEMA value is default
Param TGT_VOLUME value is psftos
Param FOLDER_NAME value is bronze
Param TBL_LIST value is 


In [20]:
if tgt_type == "VOL": 
	BronzeTablesToVol(
    	src_catalog=src_catalog,
    	src_schema=src_schema,
    	tgt_catalog=tgt_catalog,
    	tgt_schema=tgt_schema,
    	tgt_volume=tgt_volume,
   	 	folder_name=folder_name,
    	tbl_list=tbl_list
	)

Table copied to=/Volumes/default/default/psftos/bronze/ps_payer


Table copied to=/Volumes/default/default/psftos/bronze/ps_clm_hdr


Table copied to=/Volumes/default/default/psftos/bronze/ps_clm_line


Table copied to=/Volumes/default/default/psftos/bronze/ps_member


Table copied to=/Volumes/default/default/psftos/bronze/ps_provider


Table copied to=/Volumes/default/default/psftos/bronze/ps_clm_status_hist


Table copied to=/Volumes/default/default/psftos/bronze/director
